# 🏗️ CASMI 2026: V17 Precomputed Artifact Builder (Offline Engine)
### *Precomputes and packages all GBDT models, 712k candidate pool, and analog index on Kaggle Cloud*

**Outputs generated in `/kaggle/working`:**
1. `rankers_v17_ensemble.joblib` (~2-5 MB): 8 Calibrated HistGradientBoosting rankers pre-fitted on `rank_train.npz`.
2. `pool_712k_precomputed.npz` (~70-90 MB): 712,199 unified candidate structures with bitpacked 6,930-bit fingerprints pre-generated.
3. `rep_analog_scaffolds.npz` (~50-80 MB): Representative analog spectral index for instant similarity queries.

> **Usage in Inference Notebook:** Attach this notebook as input (`kernel_sources`). The inference notebook loads all assets in **< 3 seconds**!


In [ ]:
import os, sys, glob, time, pickle, math, subprocess
import joblib
from multiprocessing import Pool as MPool
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import pyarrow as pa
from numba import njit, prange
from sklearn.ensemble import HistGradientBoostingClassifier

T0 = time.time()

def find_file(name):
    hits = glob.glob(f'/kaggle/input/**/{name}', recursive=True)
    if not hits:
        raise FileNotFoundError(f"File not found in /kaggle/input: {name}")
    return sorted(hits, key=len)[0]

COMP   = os.path.dirname(find_file('test.parquet'))
TRAIN  = os.path.join(COMP, 'train.parquet')
TEST   = os.path.join(COMP, 'test.parquet')

print(f"[INFO] Competition data found in: {COMP}")

# Install offline RDKit wheel
rdkit_whls = glob.glob('/kaggle/input/**/rdkit-*.whl', recursive=True)
if rdkit_whls:
    print(f"[INFO] Installing offline RDKit wheel: {rdkit_whls[0]}")
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-index', rdkit_whls[0]], check=False)

from rdkit import Chem, RDLogger
from rdkit.Chem import rdFingerprintGenerator, MACCSkeys
from rdkit.Chem.Descriptors import ExactMolWt
RDLogger.DisableLog('rdApp.*')
print("[SUCCESS] RDKit successfully loaded.")


In [ ]:
# ===================================================================================
# 1. FIT AND EXPORT 8 GBDT RANKERS (2 Priors x 4 Seeds)
# ===================================================================================
t_start_gbdt = time.time()
rank_train_path = find_file('rank_train.npz')
z = np.load(rank_train_path)
NFEAT = z['X'].shape[1]

W1_PRIORS = (0.30, 0.60)
SEEDS     = (0, 1, 2, 3)
GBM_CONFIG = dict(max_depth=6, max_iter=500, learning_rate=0.03,
                  min_samples_leaf=80, l2_regularization=1.0)

RANKERS = []
for w1 in W1_PRIORS:
    W = np.where(z['M'] == 0, w1, 1.0 - w1)
    for sd in SEEDS:
        m = HistGradientBoostingClassifier(random_state=sd, **GBM_CONFIG)
        m.fit(z['X'], z['Y'], sample_weight=W)
        RANKERS.append(m)

out_rankers_path = '/kaggle/working/rankers_v17_ensemble.joblib'
joblib.dump(RANKERS, out_rankers_path)
print(f"[SUCCESS] 8 GBDT Rankers trained and saved to {out_rankers_path} ({time.time()-t_start_gbdt:.1f}s)")
print(f"          File size: {os.path.getsize(out_rankers_path) / (1024*1024):.2f} MB")


In [ ]:
# ===================================================================================
# 2. PRECOMPUTE AND EXPORT UNIFIED 712K CANDIDATE POOL
# ===================================================================================
t_start_pool = time.time()

class Pool:
    def __init__(self, fp, mass, keys, smiles, nbits):
        self.fp = fp; self.mass = mass
        self.keys = keys; self.smiles = smiles; self.nbits = nbits

# Load COCONUT components
co_mass = np.load(find_file('coco_mass.npy'))
co_fp   = np.load(find_file('coco_fp.npy'))
with open(find_file('coco_meta.pkl'), 'rb') as f:
    cm = pickle.load(f)
print(f"[INFO] COCONUT 2.0: {len(co_mass):,} structures loaded ({time.time()-t_start_pool:.1f}s)")

# Fingerprint generator for training structures
SEL_BITS = np.load(find_file('fp_bits.npy'))
FP_GENS = [
    rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=4096),
    rdFingerprintGenerator.GetMorganGenerator(radius=3, fpSize=4096),
    rdFingerprintGenerator.GetRDKitFPGenerator(fpSize=2048),
]

def fp_and_mass(smi):
    m = Chem.MolFromSmiles(smi)
    if m is None: return None
    arrs = [gen.GetFingerprintAsNumPy(m) for gen in FP_GENS]
    maccs = np.zeros(167, dtype=np.uint8)
    for idx in MACCSkeys.GenMACCSKeys(m).GetOnBits():
        maccs[idx] = 1
    arrs.append(maccs)
    raw = np.concatenate(arrs)[SEL_BITS]
    return raw, ExactMolWt(m)

# Read train structures
tr_df = pq.read_table(TRAIN, columns=['normalized_smiles', 'inchikey14']).to_pandas()
tr_unique = tr_df.drop_duplicates(subset=['inchikey14']).copy()
print(f"[INFO] Computing fingerprints for {len(tr_unique):,} training structures (MPool)...", flush=True)

with MPool(os.cpu_count()) as mp:
    res = mp.map(fp_and_mass, list(tr_unique.normalized_smiles), chunksize=500)

ok = [i for i, r in enumerate(res) if r is not None]
tr_fp = np.packbits(np.stack([res[i][0] for i in ok]), axis=1)
tr_mass = np.array([res[i][1] for i in ok], dtype=np.float32)
tr_keys = tr_unique.inchikey14.values[ok]
tr_smi  = tr_unique.normalized_smiles.values[ok]

# Unified Pool Assembly
all_fp = np.vstack([co_fp, tr_fp])
all_mass = np.concatenate([co_mass, tr_mass])
all_keys = np.concatenate([np.asarray(cm['keys'], dtype=object), tr_keys])
all_smis = np.concatenate([np.asarray(cm['smiles'], dtype=object), tr_smi])

good = np.isfinite(all_mass)
final_fp = all_fp[good]
final_mass = all_mass[good]
final_keys = all_keys[good]
final_smis = all_smis[good]
nbits = cm['nbits']

# Sort by exact mass for O(log N) binary search in inference
sort_idx = np.argsort(final_mass)
final_fp = final_fp[sort_idx]
final_mass = final_mass[sort_idx]
final_keys = final_keys[sort_idx]
final_smis = final_smis[sort_idx]

out_pool_path = '/kaggle/working/pool_712k_precomputed.npz'
np.savez_compressed(out_pool_path, fp=final_fp, mass=final_mass,
                    keys=final_keys, smiles=final_smis, nbits=nbits)
print(f"[SUCCESS] 712k Candidate Pool precomputed & saved to {out_pool_path} ({time.time()-t_start_pool:.1f}s)")
print(f"          Structures: {len(final_mass):,}, File size: {os.path.getsize(out_pool_path) / (1024*1024):.2f} MB")


In [ ]:
# ===================================================================================
# 3. PRECOMPUTE AND EXPORT REPRESENTATIVE ANALOG SCAFFOLD SET
# ===================================================================================
t_start_rep = time.time()

MASS = dict(C=12.0, H=1.00782503207, N=14.0030740048, O=15.9949146196, Na=22.9897692809, K=38.96370668, Cl=34.96885268)
E = 0.00054857990; PROTON = MASS['H'] - E
H2O = 2*MASS['H'] + MASS['O']; NH4 = MASS['N'] + 4*MASS['H']
FORMATE = MASS['C'] + 2*MASS['H'] + 2*MASS['O']; ACETATE = 2*MASS['C'] + 4*MASS['H'] + 2*MASS['O']

ADDUCTS = {
    '[M+H]+':(1,1,PROTON), '[M+NH4]+':(1,1,NH4-E), '[M+Na]+':(1,1,MASS['Na']-E), '[M+K]+':(1,1,MASS['K']-E),
    '[M-H2O+H]+':(1,1,PROTON-H2O), '[M-2H2O+H]+':(1,1,PROTON-2*H2O), '[M+2H]2+':(1,2,2*PROTON), '[M]+':(1,1,-E),
    '[M-H2O]+':(1,1,-E-H2O), '[M+CH3OH+H]+':(1,1,PROTON+MASS['C']+4*MASS['H']+MASS['O']),
    '[M+CH3CN+H]+':(1,1,PROTON+2*MASS['C']+3*MASS['H']+MASS['N']),
    '[M-H]-':(1,1,-PROTON), '[M-H2O-H]-':(1,1,-PROTON-H2O), '[M+CH2O2-H]-':(1,1,FORMATE-PROTON),
    '[M+C2H4O2-H]-':(1,1,ACETATE-PROTON), '[M+Cl]-':(1,1,MASS['Cl']+E), '[M]-':(1,1,E), '[M-2H]-':(1,2,-2*PROTON),
    '[M+Na-2H]-':(1,1,MASS['Na']-2*PROTON), '[2M+H]+':(2,1,PROTON), '[2M+Na]+':(2,1,MASS['Na']-E),
    '[2M+NH4]+':(2,1,NH4-E), '[2M-H]-':(2,1,-PROTON), '[2M+K]+':(2,1,MASS['K']-E),
    '[2M+CH2O2-H]-':(2,1,FORMATE-PROTON), '[2M+C2H4O2-H]-':(2,1,ACETATE-PROTON), '[2M+Na-2H]-':(2,1,MASS['Na']-2*PROTON),
}

def neutral_mass(prec, adducts):
    out = np.full(len(prec), np.nan, np.float64)
    for i, (p, a) in enumerate(zip(prec, adducts)):
        if a in ADDUCTS:
            n, z, d = ADDUCTS[a]
            out[i] = (p * z - d) / n
    return out

tr_full = pq.read_table(TRAIN, columns=['precursor_mz', 'adduct', 'inchikey14', 'ms2_mzs', 'ms2_normalized_intensities']).to_pandas()
tr_full['nm'] = neutral_mass(tr_full.precursor_mz.values.astype(np.float64), tr_full.adduct.values)
tr_valid = tr_full[np.isfinite(tr_full['nm'])].copy()

# Select longest/most representative spectrum per unique InChIKey14
def pick_rep(group):
    lengths = group['ms2_mzs'].map(len)
    best_idx = lengths.idxmax()
    return group.loc[best_idx]

print(f"[INFO] Selecting representative spectra across unique structures...", flush=True)
rep_df = tr_valid.groupby('inchikey14', as_index=False).apply(pick_rep)

# Sort by neutral mass
rep_df = rep_df.sort_values('nm').reset_index(drop=True)
rep_nm = rep_df['nm'].values.astype(np.float32)
rep_key = rep_df['inchikey14'].values

# Pack spectra arrays
rep_mz_list = list(rep_df['ms2_mzs'])
rep_it_list = list(rep_df['ms2_normalized_intensities'])
lens = [len(m) for m in rep_mz_list]
rep_off = np.zeros(len(lens) + 1, dtype=np.int64)
rep_off[1:] = np.cumsum(lens)
rep_mz = np.concatenate(rep_mz_list).astype(np.float32) if rep_mz_list else np.empty(0, np.float32)
rep_it = np.concatenate(rep_it_list).astype(np.float32) if rep_it_list else np.empty(0, np.float32)

out_rep_path = '/kaggle/working/rep_analog_scaffolds.npz'
np.savez_compressed(out_rep_path, rep_nm=rep_nm, rep_key=rep_key, rep_off=rep_off,
                    rep_mz=rep_mz, rep_it=rep_it)
print(f"[SUCCESS] Analog Representative scaffolds precomputed & saved to {out_rep_path} ({time.time()-t_start_rep:.1f}s)")
print(f"          Unique scaffolds: {len(rep_nm):,}, File size: {os.path.getsize(out_rep_path) / (1024*1024):.2f} MB")


In [ ]:
# ===================================================================================
# SUMMARY OF PRECOMPUTED ARTIFACTS GENERATED IN /kaggle/working
# ===================================================================================
print('='*70)
print('ALL PRECOMPUTED ARTIFACTS READY IN /kaggle/working:')
for f in sorted(os.listdir('/kaggle/working')):
    fp = os.path.join('/kaggle/working', f)
    if os.path.isfile(fp):
        sz_mb = os.path.getsize(fp) / (1024 * 1024)
        print(f'  ✓ {f:<35} | {sz_mb:>8.2f} MB')
print('='*70)
print(f'Total precomputation runtime: {(time.time() - T0)/60:.1f} minutes.')
